In [36]:
import anndata as ad
import pandas as pd
import numpy as np
import os

# Get FP

In [37]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smile in smiles_list:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            fps.append(None)
        else:
            fp = gen.GetFingerprint(mol)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
    return fps

In [38]:
de_train = ad.read_h5ad('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')
de_test = ad.read_h5ad('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/de_test.h5ad')

In [39]:
de = ad.concat([de_train, de_test])
sm_smiles = de.obs[['sm_name', 'SMILES']].drop_duplicates()
sm_smiles['ECFP:2'] = smiles_to_fingerprints(sm_smiles['SMILES'])
sm_smiles = sm_smiles.rename(columns = {'SMILES': 'smiles', 'sm_name': 'perturbagen'})


In [40]:
sm_smiles

,perturbagen,smiles,ECFP:2
"NK cells, TIE2 Kinase Inhibitor",TIE2 Kinase Inhibitor,COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"B cells, MK-5108",MK-5108,O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, Lapatinib",Lapatinib,CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"B cells, Belinostat",Belinostat,O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"B cells, Dabrafenib",Dabrafenib,CC(C)(C)c1nc(-c2cccc(NS(=O)(=O)c3c(F)cccc3F)c2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...
"B cells, GLPG0634",GLPG0634,O=C(Nc1nc2cccc(-c3ccc(CN4CCS(=O)(=O)CC4)cc3)n2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, Mubritinib (TAK 165)",Mubritinib (TAK 165),FC(F)(F)c1ccc(/C=C/c2nc(COc3ccc(CCCCn4ccnn4)cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, Vanoxerine",Vanoxerine,Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, SB525334",SB525334,Cc1cccc(-c2[nH]c(C(C)(C)C)nc2-c2ccc3nccnc3c2)n1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


# Get Pubchem

In [41]:
op3 = ad.read_h5ad('../../../data/op3/pseudobulk_processed/sep_rep/op3_standardized_processed.h5ad')
op3_obs = op3.obs
df_op3_sm = op3_obs.drop_duplicates(['perturbagen', 'pubchem_cid'])[['perturbagen', 'pubchem_cid']].reset_index(drop=True)

In [42]:
df_op3_sm_add = pd.DataFrame({'perturbagen': ['Belinostat', 'Dabrafenib'],
                              'pubchem_cid': [6918638, 44462760]})

In [43]:
df_emb_op3 = pd.concat([df_op3_sm, df_op3_sm_add]).reset_index(drop=True)

In [44]:
df_emb_op3['pubchem_cid'] = df_emb_op3['pubchem_cid'].astype(str)

In [45]:
df_emb_op3

,perturbagen,pubchem_cid
0,TIE2 Kinase Inhibitor,23625762
1,MK-5108,24748204
2,Lapatinib,208908
3,Dimethyl Sulfoxide,679
4,Atorvastatin,60823
...,...,...
136,Vanoxerine,3455
137,SB525334,9967941
138,HYDROXYUREA,3657
139,Belinostat,6918638


# Get embeddings and JOIN

In [1]:
epoch_dirs = ['epoch_epoch_0000', 'epoch_epoch_0004', 'epoch_epoch_0009', 'epoch_epoch_0014', 'epoch_epoch_0019', 'epoch_epoch_0024']

In [47]:
df_emb_op3

,perturbagen,pubchem_cid
0,TIE2 Kinase Inhibitor,23625762
1,MK-5108,24748204
2,Lapatinib,208908
3,Dimethyl Sulfoxide,679
4,Atorvastatin,60823
...,...,...
136,Vanoxerine,3455
137,SB525334,9967941
138,HYDROXYUREA,3657
139,Belinostat,6918638


In [48]:
import os
os.makedirs('../../data/benchmark/resources/datasets/neurips-2023-data-subsample', exist_ok=True)

for epoch_dir in epoch_dirs:
    df_pert = pd.read_pickle(f'../../../lpm_style/files/single_run/{epoch_dir}/df_pert.pkl')
    df_emb_op3_ = df_emb_op3.merge(df_pert, left_on='pubchem_cid', right_on='symbol')
    df_emb_op3_merged = df_emb_op3_.merge(sm_smiles, how='left', left_on='perturbagen', right_on='perturbagen').rename(columns={'lpm_style_embeddings': 'LPM_emb'})[['perturbagen', 'LPM_emb', 'smiles', 'ECFP:2']].reset_index(drop=True)
    tag = int(epoch_dir.split('_')[-1])
    df_emb_op3_merged.to_pickle(f'../../data/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb_{tag + 1}.pkl')

# SPLIT

In [49]:
import anndata as ad
import pandas as pd
import pickle

In [50]:
import numpy as np

In [51]:
ratio = 0.25

In [52]:
op3_train_subsample = ad.read_h5ad('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')
op3_test_subsample = ad.read_h5ad('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/de_test.h5ad')
df = pd.read_csv('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/id_map.csv')

In [85]:
path = '../../data/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb_5.pkl'
with open(path, 'rb') as fp:
    op3_emb = pickle.load(fp)

In [84]:
op3_emb

,perturbagen,LPM_emb,smiles,ECFP:2
0,TIE2 Kinase Inhibitor,"[-0.13941386342048645, 0.16943304240703583, -0...",COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,MK-5108,"[0.16495119035243988, 0.033752117305994034, 0....",O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Lapatinib,"[-0.43026700615882874, 0.042399924248456955, -...",CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Atorvastatin,"[-0.41252362728118896, 0.061428770422935486, -...",CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Ganetespib (STA-9090),"[-0.018711091950535774, -0.22836847603321075, ...",CC(C)c1cc(-c2n[nH]c(=O)n2-c2ccc3c(ccn3C)c2)c(O...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
133,Vanoxerine,"[0.023566026240587234, 0.12518268823623657, -0...",Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
134,SB525334,"[0.22825351357460022, -0.16349823772907257, -0...",Cc1cccc(-c2[nH]c(C(C)(C)C)nc2-c2ccc3nccnc3c2)n1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
135,HYDROXYUREA,"[-0.2142648547887802, 0.3217810094356537, -0.0...",NC(O)=NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
136,Belinostat,"[-0.07541624456644058, 0.05108083412051201, -0...",O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [86]:
op3_emb

,perturbagen,LPM_emb,smiles,ECFP:2
0,TIE2 Kinase Inhibitor,"[0.008914132602512836, -0.2767038345336914, 0....",COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,MK-5108,"[-0.1451248675584793, -0.1409549117088318, -0....",O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Lapatinib,"[0.25274351239204407, 0.11319196224212646, 0.0...",CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Atorvastatin,"[-0.017614640295505524, -0.0697387084364891, 0...",CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Ganetespib (STA-9090),"[0.04162070155143738, 0.5718062520027161, -0.2...",CC(C)c1cc(-c2n[nH]c(=O)n2-c2ccc3c(ccn3C)c2)c(O...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
133,Vanoxerine,"[-0.03823103383183479, -0.13062560558319092, -...",Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
134,SB525334,"[-0.023722713813185692, 0.0979263111948967, 0....",Cc1cccc(-c2[nH]c(C(C)(C)C)nc2-c2ccc3nccnc3c2)n1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
135,HYDROXYUREA,"[-0.48559603095054626, 0.38605403900146484, -0...",NC(O)=NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
136,Belinostat,"[0.0034370599314570427, -0.1233164444565773, -...",O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [54]:
op3_train_subsample = op3_train_subsample[op3_train_subsample.obs['sm_name'].isin(op3_emb['perturbagen'])].copy()
op3_test_subsample = op3_test_subsample[op3_test_subsample.obs['sm_name'].isin(op3_emb['perturbagen'])].copy()

In [55]:
op3_test_subsample

AnnData object with n_obs × n_vars = 149 × 5317
    obs: 'sm_cell_type', 'cell_type', 'sm_name', 'sm_lincs_id', 'SMILES', 'split', 'control'
    uns: 'dataset_description', 'dataset_id', 'dataset_name', 'dataset_organism', 'dataset_reference', 'dataset_summary', 'dataset_url', 'single_cell_obs'
    layers: 'AveExpr', 'B', 'P.Value', 'adj.P.Value', 'clipped_sign_log10_pval', 'is_de', 'is_de_adj', 'logFC', 'sign_log10_adj_pval', 'sign_log10_pval', 't'

In [56]:
op3_subsample = ad.concat([op3_train_subsample, op3_test_subsample], uns_merge='same')

In [57]:
df_single_cell_obs = pd.concat([op3_train_subsample.uns['single_cell_obs'], op3_test_subsample.uns['single_cell_obs']])

In [58]:
compounds = np.array(op3_subsample.obs['sm_name'].unique())

In [59]:
np.random.seed(42)
test_sample = np.random.choice(compounds, size=int(len(compounds) * ratio), replace=False)

In [60]:
op3_subsample.obs['new_split'] = np.where(op3_subsample.obs['sm_name'].isin(test_sample), 'test', 'train')

In [61]:
op3_train_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'train'].copy()
op3_train_subsample_.uns['single_cell_obs'] = df_single_cell_obs[~df_single_cell_obs['sm_name'].isin(test_sample)]
op3_train_subsample_.write_h5ad('../../data/benchmark/resources/datasets/neurips-2023-data-subsample/de_train.h5ad', compression='gzip')

In [62]:
op3_test_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'test'].copy()
op3_test_subsample_.uns['single_cell_obs'] = df_single_cell_obs[df_single_cell_obs['sm_name'].isin(test_sample)]
op3_test_subsample_.write_h5ad('../../data/benchmark/resources/datasets/neurips-2023-data-subsample/de_test.h5ad', compression='gzip')

In [63]:
op3_test_subsample_.obs[['sm_name', 'cell_type']].reset_index(drop=True).reset_index().rename(columns={'index': 'id'}).to_csv('../../data/benchmark/resources/datasets/neurips-2023-data-subsample/id_map.csv', index=False)

# Check emb

In [64]:
op3_train_subsample_

AnnData object with n_obs × n_vars = 411 × 5317
    obs: 'sm_cell_type', 'cell_type', 'sm_name', 'sm_lincs_id', 'SMILES', 'split', 'control', 'new_split'
    uns: 'dataset_description', 'dataset_id', 'dataset_name', 'dataset_organism', 'dataset_reference', 'dataset_summary', 'dataset_url', 'single_cell_obs'
    layers: 'AveExpr', 'B', 'P.Value', 'adj.P.Value', 'clipped_sign_log10_pval', 'is_de', 'is_de_adj', 'logFC', 'sign_log10_adj_pval', 'sign_log10_pval', 't'

In [65]:
adata_train_prev = ad.read_h5ad('../../data_lpm_stype_epoch1/benchmark/resources/datasets/neurips-2023-data-subsample/de_train.h5ad')

In [66]:
adata_test_prev = ad.read_h5ad('../../data_lpm_stype_epoch1/benchmark/resources/datasets/neurips-2023-data-subsample/de_test.h5ad')

In [68]:
op3_pickle = pd.read_pickle('../../data_lpm_stype_epoch1/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb.pkl')

In [69]:
(op3_test_subsample_.layers['clipped_sign_log10_pval'] == adata_test_prev.layers['clipped_sign_log10_pval']).all()

True

In [75]:
(op3_train_subsample_.layers['clipped_sign_log10_pval'] == adata_train_prev.layers['clipped_sign_log10_pval']).all()

True

In [82]:
(op3_emb[['ECFP:2']].astype(str) == op3_pickle[['ECFP:2']].astype(str)).all()

ECFP:2    True
dtype: bool

In [74]:
op3_emb[['perturbagen', 'smiles', ]].compare(op3_pickle[['perturbagen', 'smiles']])

Empty DataFrame
Columns: []
Index: []